In [13]:
!pip install pm4py

In [14]:
import pandas as pd

from pm4py.objects.log.util import dataframe_utils
from pm4py.objects.conversion.log import converter as log_converter

from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.algo.evaluation.replay_fitness import algorithm as fitness_evaluator

from pm4py.statistics.traces.generic.log import case_statistics
from pm4py.objects.conversion.process_tree import converter as pt_converter


# ------- Petri Net Visualization Fallback -------
try:
    from pm4py.visualization.petri_net import visualize as pn_vis_factory
except:
    from pm4py.visualization.petri_net.common import visualize as pn_vis_factory

In [15]:
# 1) Load CSV
df = pd.read_csv("turnaround_log.csv")

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])
df = dataframe_utils.convert_timestamp_columns_in_df(df)

log = log_converter.apply(df)

In [16]:
# 2) Discover process as Process Tree first
tree = inductive_miner.apply(
    log,
    variant=inductive_miner.Variants.IMf
)

# 3) Convert Process Tree -> Petri Net
net, initial_marking, final_marking = pt_converter.apply(tree)

In [17]:
# 4) Visualize Petri net
gviz = pn_vis_factory.apply(net, initial_marking, final_marking)

gviz.save("petri_net.dot")
print("Petri net model saved as petri_net.dot (Graphviz source file)")

gviz.save("petri_net.png")
print("Petri net saved as petri_net.png")


Petri net model saved as petri_net.dot (Graphviz source file)
Petri net saved as petri_net.png


In [19]:
# 5) Conformance checking
aligned = alignments.apply_log(log, net, initial_marking, final_marking)

# Import replay fitness (VERY important)
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness

print("Computing replay fitness...")

# Compute fitness using the correct signature for your PM4Py version
fitness = replay_fitness.apply(
    log,
    net,
    initial_marking,
    final_marking,
    variant=replay_fitness.Variants.ALIGNMENT_BASED
)

print("\n=== FITNESS RESULTS ===")
print("Full fitness dict:", fitness)

if isinstance(fitness, dict) and "average_trace_fitness" in fitness:
    print("Average trace fitness:", fitness["average_trace_fitness"])


aligning log, completed variants :: 100%|██████████| 9/9 [00:00<00:00, 20.84it/s]


Computing replay fitness...


aligning log, completed variants :: 100%|██████████| 9/9 [00:00<00:00, 20.50it/s]


=== FITNESS RESULTS ===
Full fitness dict: {'percFitTraces': 100.0, 'averageFitness': 1.0, 'percentage_of_fitting_traces': 100.0, 'average_trace_fitness': 1.0, 'log_fitness': 0.9999499927985936}
Average trace fitness: 1.0


In [20]:
print("\nComputing turnaround performance statistics...")

case_durations = case_statistics.get_all_case_durations(log)

# Handle the case: old PM4Py returns list of floats
if isinstance(case_durations, list) and isinstance(case_durations[0], float):
    durations_list = case_durations
else:
    # fallback if dictionary format
    durations_list = [x["caseDuration"] for x in case_durations]

durations_list_sorted = sorted(durations_list, reverse=True)

num_cases = len(durations_list_sorted)
avg_tat = sum(durations_list_sorted) / num_cases

print("\n=== PERFORMANCE RESULTS ===")
print("Number of turnarounds (cases):", num_cases)
print("Average turnaround time (seconds):", avg_tat)
print("Average turnaround time (minutes):", avg_tat / 60)

print("\nTop 5 slowest turnarounds (seconds):")
for d in durations_list_sorted[:5]:
    print(d)


Computing turnaround performance statistics...

=== PERFORMANCE RESULTS ===
Number of turnarounds (cases): 30550
Average turnaround time (seconds): 67522.6271685761
Average turnaround time (minutes): 1125.3771194762683

Top 5 slowest turnarounds (seconds):
85980.0
85920.0
85920.0
85800.0
85800.0
